In [1]:
from glob import glob
import pandas as pd
import os
import soundfile as sf
from tqdm import tqdm
from multiprocess import Pool
import librosa
import itertools
import io
import numpy as np
import json

def chunks(l, n):
    for i in range(0, len(l), n):
        yield (l[i: i + n], i // n)

def multiprocessing(strings, function, cores=6, returned=True):
    df_split = chunks(strings, len(strings) // cores)
    pool = Pool(cores)
    pooled = pool.map(function, df_split)
    pool.close()
    pool.join()

    if returned:
        return list(itertools.chain(*pooled))

In [2]:
files = glob('libritts_r_filtered/*/*.parquet')
len(files)

197

In [8]:
def loop(files):

    os.environ['OMP_NUM_THREADS'] = '1'
    os.environ['OPENBLAS_NUM_THREADS'] = '1'
    
    files, _ = files

    data = []
    for f in tqdm(files):
        base = '_'.join(f.split('/')[:2])
        f_new = f.replace('/', '-').replace('.parquet', '')
        os.makedirs(base, exist_ok=True)
        df = pd.read_parquet(f)
        for i in range(len(df)):
            t = df['text_original'].iloc[i].strip()
            if len(t) < 2:
                continue
            audio_filename = f'{f_new}_{i}.mp3'
            audio_filename = os.path.join(base, audio_filename)
            b = df['audio'].iloc[i]['bytes']
            audio_np, sr = sf.read(io.BytesIO(b))
            if audio_np.ndim > 1:
                audio_np = audio_np.mean(axis=1)
            if audio_np.shape[0] < 10000:
                continue
            sf.write(audio_filename, audio_np, sr)
            
            data.append({
                'audio_filename': audio_filename,
                'text': t,
                'speaker': f"{base}_{df['speaker_id'].iloc[i]}"
            })
        
    return data

In [9]:
# data = loop((files[:1], 0))
# len(data)

In [ ]:
data = multiprocessing(files, loop, cores = min(30, len(files)))

  0%|          | 0/6 [00:00<?, ?it/s]

In [11]:
len(data)

358169

In [12]:
data[0]

{'audio_filename': 'libritts_r_filtered_other/libritts_r_filtered-other-train.other.500-00055-of-00102_0.mp3',
 'text': 'Where could he be?',
 'speaker': 'libritts_r_filtered_other_428'}

In [13]:
from datasets import Dataset

dataset = Dataset.from_list(data)
dataset[0]

/home/ubuntu/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


{'audio_filename': 'libritts_r_filtered_other/libritts_r_filtered-other-train.other.500-00055-of-00102_0.mp3',
 'text': 'Where could he be?',
 'speaker': 'libritts_r_filtered_other_428'}

In [14]:
dataset.push_to_hub('malaysia-ai/Multilingual-TTS', 'libritts_r_filtered')

Creating parquet from Arrow format: 100%|██████████| 1/1 [00:00<00:00,  6.47ba/s]
Processing Files (0 / 0): |          |  0.00B /  0.00B            
Processing Files (0 / 1): 100%|█████████▉| 25.6MB / 25.7MB,   ???B/s  
Processing Files (1 / 1): 100%|██████████| 25.7MB / 25.7MB,  381kB/s  
Processing Files (1 / 1): 100%|██████████| 25.7MB / 25.7MB,  191kB/s  
New Data Upload: 100%|██████████| 16.4MB / 16.4MB,  191kB/s  
Uploading the dataset shards: 100%|██████████| 1/1 [00:01<00:00,  1.14s/ shards]


CommitInfo(commit_url='https://huggingface.co/datasets/malaysia-ai/Multilingual-TTS/commit/a2b76ff03e6159822d99c744ddbaa062fc7782b1', commit_message='Upload dataset', commit_description='', oid='a2b76ff03e6159822d99c744ddbaa062fc7782b1', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/malaysia-ai/Multilingual-TTS', endpoint='https://huggingface.co', repo_type='dataset', repo_id='malaysia-ai/Multilingual-TTS'), pr_revision=None, pr_num=None)

In [15]:
audio_files = [d['audio_filename'] for d in data]

with open('libritts_r_filtered-audio.json', 'w') as fopen:
    json.dump(list(set(audio_files)), fopen)

In [17]:
folders = glob('libritts_r_filtered_*')
folders = [f for f in folders if '.zip' not in f]
for f in folders:
    print(f)
    os.system(f'zip -rq {f}.zip {f}')

libritts_r_filtered_other
libritts_r_filtered_clean_neucodec
libritts_r_filtered_other_neucodec
libritts_r_filtered_clean


In [18]:
from huggingface_hub import HfApi
api = HfApi()

for f in glob('libritts_r_filtered_*.zip'):
    api.upload_file(
        path_or_fileobj=f,
        path_in_repo=f,
        repo_id="malaysia-ai/Multilingual-TTS",
        repo_type="dataset",
    )

Processing Files (0 / 0): |          |  0.00B /  0.00B            
Processing Files (0 / 1):   0%|          | 13.6MB / 7.59GB, 68.3MB/s  
Processing Files (0 / 1):   2%|▏         |  141MB / 7.59GB,  352MB/s  
Processing Files (0 / 1):   4%|▎         |  277MB / 7.59GB,  461MB/s  
Processing Files (0 / 1):   5%|▌         |  398MB / 7.59GB,  498MB/s  
Processing Files (0 / 1):   7%|▋         |  508MB / 7.59GB,  508MB/s  
Processing Files (0 / 1):   8%|▊         |  612MB / 7.59GB,  510MB/s  
Processing Files (0 / 1):  10%|▉         |  739MB / 7.59GB,  528MB/s  
Processing Files (0 / 1):  11%|█         |  801MB / 7.59GB,  501MB/s  
Processing Files (0 / 1):  12%|█▏        |  906MB / 7.59GB,  504MB/s  
Processing Files (0 / 1):  13%|█▎        | 1.01GB / 7.59GB,  507MB/s  
Processing Files (0 / 1):  15%|█▌        | 1.16GB / 7.59GB,  525MB/s  
Processing Files (0 / 1):  17%|█▋        | 1.29GB / 7.59GB,  537MB/s  
Processing Files (0 / 1):  19%|█▊        | 1.41GB / 7.59GB,  541MB/s  
Processing